# Trading Performance Diagnostics

**Question:** can we separate an execution-cost problem from signal deterioration?

All trading records, quotes, signals and incidents are simulated. Lagged market volatility is the only real market input. This is a reproducible engineering/statistical experiment, not a live trading result.

Read the detailed methods in [METHODOLOGY.md](../research/strategist_lab/METHODOLOGY.md).

## 1. Verify frozen inputs

The loader checks file hashes, schemas, dates and decimal units. It never silently downloads a newer vintage.

In [1]:
from pathlib import Path
import os, json
if not Path('research/strategist_lab').exists():
    os.chdir('..')
assert Path('research/strategist_lab').exists(), 'Start Jupyter in the repository root or notebooks folder.'
from research.strategist_lab.data import load_data
from research.strategist_lab import OUTPUT
industries, factors = load_data()
print('Input window:', industries.index[0].date(), 'to', industries.index[-1].date())
print('Rows:', len(industries), '| Public data vintage and SHA-256 hashes are in data/strategist_lab/manifest.json')


Input window: 2015-01-02 to 2025-12-31
Rows: 2766 | Public data vintage and SHA-256 hashes are in data/strategist_lab/manifest.json


## 2. Run the full ledger and monitoring pipeline

The default simulation has 400 parents per session. The model sees only training features and validation errors; hidden incident labels join after scores and thresholds are frozen.

In [2]:
from research.strategist_lab.execution import ExecutionConfig, run_execution
metrics, daily, venue_monitor, signal_monitor = run_execution(factors, ExecutionConfig())
print(json.dumps({k: metrics[k] for k in ['orders','fills','train_end','validation_end','test_start','test_sessions']}, indent=2))


{
  "orders": 100000,
  "fills": 289839,
  "train_end": "2025-07-07",
  "validation_end": "2025-10-03",
  "test_start": "2025-10-06",
  "test_sessions": 61
}


## 3. Reconcile decision P&L, realised P&L and every cost

Implementation shortfall includes decision delay, market movement, filled execution, unfilled opportunity and fees. Positive cost is worse. The identities hold for both sides and for no fills.

In [3]:
print('Data-quality results:', metrics['quality'])
print('Maximum per-order errors in dollars:', metrics['reconciliation'])
print('Full-sample dollar bridge:')
print(json.dumps(metrics['components_usd'], indent=2))
assert max(metrics['reconciliation'].values()) < 1e-6


Data-quality results: {'raw_rows': 292739, 'clean_rows': 289839, 'exact_retries_removed': 2898, 'orphan_rows': 1, 'invalid_rows': 1, 'rejected_rows': 2}
Maximum per-order errors in dollars: {'max_cost_error_usd': 9.731593308970332e-10, 'max_pnl_error_usd': 8.349161362275481e-10}
Full-sample dollar bridge:
{
  "delay_usd": 676596.5483699709,
  "market_usd": 453742.78118968266,
  "execution_usd": 10237206.562399471,
  "opportunity_usd": 2241940.633955092,
  "fees": 277457.7884396389,
  "is_usd": 13886944.314353867,
  "ideal_pnl_usd": 13549910.20596359,
  "net_pnl_usd": -337034.10839028325
}


![Execution-cost bridge](../assets/images/strategist/cost-bridge.svg)

Aggregate component bps share the intended decision-notional denominator. Regression residuals instead use filled-parent execution costs.

## 4. Inspect group monitoring and independent signal performance

In [4]:
print(venue_monitor.loc[venue_monitor.alert].head(8).to_string(index=False))
print('\nSignal-decay flags:')
print(signal_monitor.loc[signal_monitor.alert].head(8).to_string(index=False))
print('\nHeld-out confusion matrices:')
print(json.dumps(metrics['alert_evaluation'], indent=2))


      date   venue  residual_bps  latency_ms  observed_bps  expected_bps  orders      score  alert  incident
2025-11-04 Venue B      5.993925  193.599338     10.568328      4.574403     131 106.839254   True      True
2025-11-05 Venue B      6.009087  194.475059     10.589625      4.580538     122 107.109616   True      True
2025-11-06 Venue B      6.011392  194.484123     10.562497      4.551105     137 107.150722   True      True
2025-11-07 Venue B      6.007564  194.205559     10.623213      4.615649     130 107.082466   True      True
2025-11-10 Venue B      5.987552  195.278962     10.292489      4.304937     139 106.725621   True      True
2025-11-11 Venue B      6.020401  194.052812     10.348874      4.328474     151 107.311356   True      True
2025-11-12 Venue B      5.963028  194.905990     10.262631      4.299602     129 106.288319   True      True
2025-11-13 Venue B      5.880993  195.400115     10.226982      4.345989     143 104.825496   True      True

Signal-decay flags

![Production monitor](../assets/images/strategist/execution-monitor.svg)

![Signal calibration](../assets/images/strategist/signal-decay.svg)

Cost and latency move together for the routing incident. A separate forecast-error series identifies the signal episode. Correlation between diagnostics supports investigation; it is not causal identification.

## 5. Challenge strong results

Read all 15 independent-seed/effect-size experiments. At zero cost shift, a flag is a false cost alarm. A strong seeded case does not establish real-world precision. Regenerate the grid using `python -m research.strategist_lab.robustness`.

In [5]:
import pandas as pd
sensitivity = pd.read_csv(OUTPUT / 'execution/sensitivity.csv')
print(sensitivity.groupby('shift_bps')[['tp','fp','fn','tn']].sum().to_string())
print('\nFive-session block-bootstrap venue intervals:')
print(pd.DataFrame(metrics['venue_intervals']).to_string(index=False))


           tp  fp  fn   tn
shift_bps                 
0.00        0   0  45  504
0.25       10   0  35  504
0.50       44   0   1  504
1.00       45   0   0  504
6.00       45   0   0  504

Five-session block-bootstrap venue intervals:
  venue  residual_mean_bps    ci_low   ci_high
Venue A           0.004874 -0.005578  0.013928
Venue B           1.462554  0.377284  3.039484
Venue C          -0.023467 -0.038441 -0.014863


## 6. Management interpretation and next experiment

Investigate the execution path and signal calibration separately. Raw venue rankings would confound order mix; conditional residuals remain descriptive unless routing assignment is exogenous.

A stronger next test uses independently sourced executions, true event/receipt times, changing regimes, persistence rules and a precommitted false-alert policy. The synthetic benchmark is a post-trade diagnostic and is not an executable price forecast.